# Intermediate Machine Learning

[Course Link](https://www.kaggle.com/learn/intermediate-machine-learning)

In [32]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import cross_val_score

from xgboost import XGBRegressor

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline

In [25]:
CWD = os.getcwd()
DATA = CWD + '/tutorials_data/'

TRAIN = DATA + 'housing_train.csv'
TEST = DATA + 'housing_test.csv'
MELB = DATA + 'housing_melbourne.csv'
AER = DATA + 'credit_card_data.csv'

## 1. Introduction

In this course, you will learn to:
- Tackle data types in real-world datasets (missing values, categorical variables)
- Design pipelines to improve quality of machine learning code
- Use advanced techniques for model validation (cross-validation)
- Build models used in competitions (XGBoost)
- Avoid common data science mistakes (leakage)

### Format Data

In [11]:
# Load data
df_train = pd.read_csv(TRAIN)
df_test = pd.read_csv(TEST)

# Extract target
y = df_train['SalePrice']
# Define features
features = ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']
# Extract features
X = df_train[features].copy()
# Extract test feature data
X_test = df_test[features].copy()

# Split training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state = 0)

# Show data
X_train.head()

,LotArea,YearBuilt,1stFlrSF,2ndFlrSF,FullBath,BedroomAbvGr,TotRmsAbvGrd
618,11694,2007,1828,0,2,3,9
870,6600,1962,894,0,1,2,5
92,13360,1921,964,0,1,2,5
817,13265,2002,1689,0,2,3,7
302,13704,2001,1541,0,2,3,6


### Define Models

In [12]:
# Define multiple RandomForest models
model_1 = RandomForestRegressor(n_estimators = 50, random_state = 0)
model_2 = RandomForestRegressor(n_estimators = 100, random_state = 0)
model_3 = RandomForestRegressor(n_estimators = 100, random_state = 0, criterion = 'absolute_error')
model_4 = RandomForestRegressor(n_estimators = 200, random_state = 0, min_samples_split = 20)
model_5 = RandomForestRegressor(n_estimators = 100, random_state = 0, max_depth = 7) 

# Define model list
list_models = [model_1, model_2, model_3, model_4, model_5]

### Define Scoring Function

In [13]:
def score_model(model, X_train = X_train, X_valid = X_valid, y_train = y_train, y_valid = y_valid):
    # Fit model
    model.fit(X_train, y_train)
    # Generate predictions
    y_predictions = model.predict(X_valid)
    
    return mean_absolute_error(y_valid, y_predictions)

### Test Models

In [14]:
# Initialise best model
best_mae = np.inf
best_model = None

# Iterate over model
for i in range(0, len(list_models)):
    # Calculate MAE
    mae = score_model(list_models[i])
    # Check for best MAE
    if mae < best_mae:
        # Assign best_mae
        best_mae = mae
        # Assign best_model
        best_model = i+1
    print(f'Model {i + 1} MAE: {mae}')

print()
print(f'Best Model: {best_model}')

Model 1 MAE: 24015.492818003917
Model 2 MAE: 23740.979228636657
Model 3 MAE: 23528.78421232877
Model 4 MAE: 23996.676789668687
Model 5 MAE: 23706.672864217904

Best Model: 3


## 2. Missing Values

There are many ways data ends up with missing values:
- A 2 bedroom house won't include value for size of a third bedroom
- A respondent may choose not to share income data

Most ML libraries, including `scikit-learn` give an error when building a model with missing values in the data. 

There are various strategies to deal with missing data.

### Drop Columns

The simplest option is to drop feature columns with missing values.

Unless most values in the dropped column are missing, them odel loses access to a lot of potentially relevant information.

### Imputation

Imputation fills missing values with a calculated value, often the **mean** of a column. 

This value won't be strictly correct in most cases; however, it leads to more accurate models than dropping columns entirely.

### Extended Imputation

Simple imputed values may be systematically above or below actual values, or rows including missing values may be unique in some other way.

In this case, the model would make better predictions by considering which values were originally missing.

In this approach, the missing values are imputed for a given column, and an additional column is created which shows the location of those imputed values.

### Example

In this example, these three methods will be tested on the Melbourne Dataset.

In [15]:
# Load data
df = pd.read_csv(MELB)

# Select target
y = df['Price']

# Select numerical features
features = df.drop(['Price'], axis = 1)
X = features.select_dtypes(exclude = ['object'])

# Divide data into train/test/validation
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state = 0)

# Show data
X_train.head()

,Rooms,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
12167,1,5.0,3182.0,1.0,1.0,1.0,0.0,NaN,1940.0,-37.85984,144.9867,13240.0
6524,2,8.0,3016.0,2.0,2.0,1.0,193.0,NaN,NaN,-37.85800,144.9005,6380.0
8413,3,12.6,3020.0,3.0,1.0,1.0,555.0,NaN,NaN,-37.79880,144.8220,3755.0
2919,3,13.0,3046.0,3.0,1.0,1.0,265.0,NaN,1995.0,-37.70830,144.9158,8870.0
6043,3,13.3,3020.0,3.0,1.0,2.0,673.0,673.0,1970.0,-37.76230,144.8272,4217.0


In [16]:
# Define scoring function
def score_dataset(X_train, X_valid, y_train, y_valid):
    # Define model
    model = RandomForestRegressor(n_estimators = 10, random_state = 0)
    # Fit model
    model.fit(X_train, y_train)
    # Generate predictions
    y_predictions = model.predict(X_valid)
    
    return mean_absolute_error(y_valid, y_predictions)

In [17]:
# Get missing value columns
list_null = [col for col in X_train.columns if X_train[col].isnull().any()]

#### Drop Columns

In [18]:
# Drop selected columns
X_train_drop = X_train.drop(list_null, axis = 1)
X_valid_drop = X_valid.drop(list_null, axis = 1)

# Calculate MAE
mae = score_dataset(X_train_drop, X_valid_drop, y_train, y_valid)

# Report
print(f'MAE w/ dropped columns: {mae:.2f}')

MAE w/ dropped columns: 183550.22


#### Simple Imputation

In [19]:
# Define imputer
my_imputer = SimpleImputer()

# Impute values
X_train_simpute = pd.DataFrame(my_imputer.fit_transform(X_train))
X_valid_simpute = pd.DataFrame(my_imputer.transform(X_valid))

# Rename columns
X_train_simpute.columns = X_train.columns
X_valid_simpute.columns = X_valid.columns

# Calculate MAE
mae = score_dataset(X_train_simpute, X_valid_simpute, y_train, y_valid)

# Report
print(f'MAE w/ simple impute: {mae:.2f}')

MAE w/ simple impute: 178166.46


#### Extended Imputation

In [20]:
# Copy data
X_train_plus = X_train.copy()
X_valid_plus = X_valid.copy()

# Generate new columns to record imputation
for column in list_null:
    X_train_plus[column + '_was_missing'] = X_train_plus[column].isnull()
    X_valid_plus[column + '_was_missing'] = X_valid_plus[column].isnull()

# Define imputer
my_imputer = SimpleImputer()
# Impute values
X_train_impute = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
X_valid_impute = pd.DataFrame(my_imputer.transform(X_valid_plus))
# Rename columns
X_train_impute.columns = X_train_plus.columns
X_valid_impute.columns = X_valid_plus.columns

# Calculate MAE
mae = score_dataset(X_train_impute, X_valid_impute, y_train, y_valid)

# Report
print(f'MAE w/ extended impute: {mae:.2f}')

MAE w/ extended impute: 178927.50


From MAE results, you can see that **Simple Imputation** performed better than **Drop Columns** and **Extended Imputation** methods.

## 3. Categorical Variables

A categorical variable takes only a limited number of values.

A survey asking how often you eat breakfast may only have responses: 
- 'Never'
- 'Rarely'
- 'Most days'
- 'Every day'

You will get an error if you try to place these variables into most ML models in Python without preprocessing.

Here, we will discuss 3 approaches for dealing with categorical variables.

### Drop Categorical Variables

The easiest approach to dealing with categorical variables is to remove them from the dataset.

This will only work well if the columns did not contain useful information.

### Ordinal Encoding

Ordinal encoding assigns each unique value to a different integer:
- 'Never' : 0
- 'Rarely' : 1
- 'Most days' : 2
- 'Every day' : 3

This approach assumes some ordering of the categories, shown above by frequency.

For tree-based models like decision trees/random forests, you can epect ordinal encoding to work well with ordinal variables.

**Note**: Training and validation data must contain the same set of categorical values in a given column for ordinal encoding to work.

### One-Hot Encoding

One-hot encoding creates new columns indicating the presence/absence of each possible value in the original data.

| Color  | Red | Yellow | Green |
|--------|-----|--------|-------|
| Red    | 1   | 0      | 0     |
| Red    | 1   | 0      | 0     |
| Yellow | 0   | 1      | 0     |
| Green  | 0   | 0      | 1     |
| Yellow | 0   | 1      | 0     |

One-hot encoding does not assume the ordering of categories, with this kind of data referred to as **nominal variables**.

One-hot encoding generally does not perform well if the categorical variables take on a large number of values.

### Example

In [21]:
# Load data
df = pd.read_csv(MELB)

# Select target
y = df['Price']
X = df.drop(columns = ['Price'])

# Train/test split
X_train_full, X_valid_full, y_train_full, y_valid_full = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state = 0)

# Define columns w/ missing values
list_null = [col for col in X_train_full.columns if X_train_full[col].isnull().any()]
# Drop columns
X_train_full.drop(columns = list_null, inplace = True)
X_valid_full.drop(columns = list_null, inplace = True)

# Select categorical columns w/ low cardinality
list_low = [col for col in X_train_full.columns if 
            X_train_full[col].nunique() < 10 and
            X_train_full[col].dtype == 'object']

# Select numerical columns
list_numerical = [col for col in X_train_full.columns if 
                  X_train_full[col].dtype in ['int64', 'float64']]

# Keep columns
list_columns = list_low + list_numerical
X_train = X_train_full[list_columns].copy()
X_valid = X_valid_full[list_columns].copy()

# Show data
X_train.head()

,Type,Method,Regionname,Rooms,Distance,Postcode,Bedroom2,Bathroom,Landsize,Lattitude,Longtitude,Propertycount
12167,u,S,Southern Metropolitan,1,5.0,3182.0,1.0,1.0,0.0,-37.85984,144.9867,13240.0
6524,h,SA,Western Metropolitan,2,8.0,3016.0,2.0,2.0,193.0,-37.85800,144.9005,6380.0
8413,h,S,Western Metropolitan,3,12.6,3020.0,3.0,1.0,555.0,-37.79880,144.8220,3755.0
2919,u,SP,Northern Metropolitan,3,13.0,3046.0,3.0,1.0,265.0,-37.70830,144.9158,8870.0
6043,h,S,Western Metropolitan,3,13.3,3020.0,3.0,1.0,673.0,-37.76230,144.8272,4217.0


In [22]:
# Get all categorical data
list_cat = [col for col in X_train.columns if X_train[col].dtype == 'object']
print(list_cat)

['Type', 'Method', 'Regionname']


In [23]:
# Define score_dataset function
def score_dataset(X_train, X_valid, y_train, y_valid):

    # Define model
    model = RandomForestRegressor(n_estimators = 100, random_state = 0)
    # Fit model
    model.fit(X_train, y_train)
    # Generate predictions
    y_pred = model.predict(X_valid)
    # Calculate MAE
    mae = mean_absolute_error(y_valid, y_pred)

    return mae

#### Drop Categorical Variables

In [24]:
# Drop categorical variables
X_train_drop = X_train.select_dtypes(exclude = ['object'])
X_valid_drop = X_valid.select_dtypes(exclude = ['object'])

# Calculate MAE
mae = score_dataset(X_train_drop, X_valid_drop, y_train, y_valid)

# Report
print(F'MAE w/ dropped categorical variables: {mae:.2f}')

MAE w/ dropped categorical variables: 175703.48


#### Ordinal Encoding

For each categorical column, a unique values are assigned a different integer.

There is an additional boost in performance if informed labels are assigned manually.

In [25]:
# Copy data
X_train_ord = X_train.copy()
X_valid_ord = X_valid.copy()

# Initialise ordinal encoder
ordinal_encoder = OrdinalEncoder()

# Apply ordinal encoding
X_train_ord[list_cat] = ordinal_encoder.fit_transform(X_train[list_cat])
X_valid_ord[list_cat] = ordinal_encoder.transform(X_valid[list_cat])

# Calculae MAE
mae = score_dataset(X_train_ord, X_valid_ord, y_train, y_valid)

# Report
print(F'MAE w/ ordinal encoding: {mae:.2f}')

MAE w/ ordinal encoding: 165936.41


#### One-Hot Encoding

In [26]:
# Initialise One Hot Encoder
onehot_encoder = OneHotEncoder(handle_unknown = 'ignore', sparse_output = False)

# Apply encoding
df_encode_train = pd.DataFrame(onehot_encoder.fit_transform(X_train[list_cat]))
df_encode_valid = pd.DataFrame(onehot_encoder.transform(X_valid[list_cat]))

# Reindex data
df_encode_train.index = X_train.index
df_encode_valid.index = X_valid.index

# Remove old categorical columns
X_train_oh = X_train.drop(columns = list_cat)
X_valid_oh = X_valid.drop(columns = list_cat)

# Add one hot encoded columns to datasets
X_train_encode = pd.concat([X_train_oh, df_encode_train], axis = 1)
X_valid_encode = pd.concat([X_valid_oh, df_encode_valid], axis = 1)

# Assert string type in all columns
X_train_encode.columns = X_train_encode.columns.astype(str)
X_valid_encode.columns = X_valid_encode.columns.astype(str)

# Calculate MAE
mae = score_dataset(X_train_encode, X_valid_encode, y_train, y_valid)

# Report
print(F'MAE w/ one hot encoding: {mae:.2f}')

MAE w/ one hot encoding: 166089.49


There doesn't seem to be any benefit to **Ordinal Encoding** vs **One Hot Encoding**, whereas both are better than the **Drop Categorical Variables** method.

## 4. Pipelines

Pipelines are a method to keep data preprocessing and modelling code organised, bindling them together to be used as a single step. This provides:

- Cleaner code
- Fewer bugs
- Easier transition to production
- More options for validation

In [32]:
# Load data
df = pd.read_csv(MELB)

# Select target
y = df['Price']
X = df.drop(columns = ['Price'])

# Train/test split
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X, y, train_size = 0.8, test_size = 0.2, random_state = 0)

# Define columns w/ missing values
list_null = [col for col in X_train_full.columns if X_train_full[col].isnull().any()]
# Drop columns
X_train_full.drop(columns = list_null, inplace = True)
X_valid_full.drop(columns = list_null, inplace = True)

# Select categorical columns w/ low cardinality
list_low = [col for col in X_train_full.columns if 
            X_train_full[col].nunique() < 10 and
            X_train_full[col].dtype == 'object']

# Select numerical columns
list_numerical = [col for col in X_train_full.columns if 
                  X_train_full[col].dtype in ['int64', 'float64']]

# Keep columns
list_columns = list_low + list_numerical
X_train = X_train_full[list_columns].copy()
X_valid = X_valid_full[list_columns].copy()

# Show data
X_train.head()

,Type,Method,Regionname,Rooms,Distance,Postcode,Bedroom2,Bathroom,Landsize,Lattitude,Longtitude,Propertycount
12167,u,S,Southern Metropolitan,1,5.0,3182.0,1.0,1.0,0.0,-37.85984,144.9867,13240.0
6524,h,SA,Western Metropolitan,2,8.0,3016.0,2.0,2.0,193.0,-37.85800,144.9005,6380.0
8413,h,S,Western Metropolitan,3,12.6,3020.0,3.0,1.0,555.0,-37.79880,144.8220,3755.0
2919,u,SP,Northern Metropolitan,3,13.0,3046.0,3.0,1.0,265.0,-37.70830,144.9158,8870.0
6043,h,S,Western Metropolitan,3,13.3,3020.0,3.0,1.0,673.0,-37.76230,144.8272,4217.0


### Example

#### Define Preprocessing Steps

We use the `ColumnTransformer` class to bundle different preprocessing steps.

In [33]:
# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy = 'constant')

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown = 'ignore'))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers = [
        ('num', numerical_transformer, list_numerical),
        ('cat', categorical_transformer, list_low)
    ]
)

#### Define The Model

In [34]:
# Define RandomForest model
model = RandomForestRegressor(n_estimators = 100, random_state = 0)

#### Create and Evaluate Pipeline

Here we use the `Pipeline` class to define the pipeline that bundles preprocessing and modelling:
- Preprocessing and model fitting happens in one line of code
- We can supply unprocessed features in X_valid to `predict()`, and the pipeline automatically preprocesses features before generating predictions

In [35]:
# Bundle preprocessing and modeling code
my_pipeline = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', model)
    ]
)

# Preprocess training data and fit model
my_pipeline.fit(X_train, y_train)

# Preprocess validation data and get predictions
y_pred = my_pipeline.predict(X_valid)

# Evaluate model
mae = mean_absolute_error(y_valid, y_pred)

# Report
print(f'MAE: {mae:.2f}')

MAE: 166089.49


Pipelines are valuable for cleaning up ML code and avoiding errors, and especially use for workflows with sophisticated data preprocessing.

## 5. Cross-Validation

Machine learning is an iterative process, requiring choices on:
- What predictive variables to use
- What types of model to use
- Which arguments to apply to those models

So far, the choices have been made in a data-driven way by measuring model quality with a validation dataset.

Given a dataset with 5000 rows, around 20% is kept as a validation dataset; however, this leaves some random chance in determining model scores.

The larger the validation set, the less randomness or noise there is in the measure of model quality, making things more reliable. 

We can only get large validation sets by removing rows from training data, which leaves smaller training datasets and worse models.

### What Is Cross-Validation?

In cross-validation, we run the modelling process on different subsets of the data to get multiple measures of model quality.

For example, the data could be divided into 5 pieces or 'folds', each 20% of the full dataset. You then run experiments on each fold, where each fold is used as a validation (or holdout) set and everything else as training data. This provides a measure of model quality based on a 20% validation set.

After 5 folds, 100% of the data has been used as validation data at some point, and model quality is assessed on all rows in the dataset, even if they are not all used simultaneously.


### When To Use Cross-Validation

Cross-validation gives a more accurate measure of model quality, important for multiple modelling decisions, but can take longer to run (with one model estimated per fold):

- For smaller datasets, where extra computational burden isn't an issue, you should run cross-validation.
- For larger datasets, a single validation set is sufficient.

There is no simple threshold for large vs small data; however, if the model takes multiple minutes to run, it may be worth switching to cross-validation.

Alternatively, cross-validation can be run and if scores for each fold seem similar, a single validation set may be sufficient.


### Example

In [37]:
# Load data
df = pd.read_csv(MELB)

# Define predictors
list_columns = ['Rooms', 'Distance', 'Landsize', 'BuildingArea', 'YearBuilt']
# Extract data
X = df[list_columns]

# Select target
y = df['Price']

In [38]:
# Define Pipeline
my_pipeline = Pipeline(
    steps = [
        ('imputer', SimpleImputer()),
        ('model', RandomForestRegressor(n_estimators = 50, random_state = 0))
    ]
)

In [46]:
# Calculate -MAE
scores = cross_val_score(
    my_pipeline, X, y,
    cv = 5,
    scoring = 'neg_mean_absolute_error'
)

# Convert to MAE
mae = scores * -1

# Report
print(f'MAE scores:\n {mae}')

MAE scores:
 [301628.7893587  303164.4782723  287298.331666   236061.84754543
 260383.45111427]


`sklearn` has a convention where all metrics are defined such that a higher number is better, and negatives allow consistency with this. Negative MAE is almost unheard of elsewhere.

Typically, a single measure of model quality is best to compare against alternative models, so the average metric across all experiments can be used.

In [48]:
# Report
print(f'Mean MAE score: {mae.mean():.2f}')

Mean MAE score: 277707.38


Hyperparameter optimisation is an important step in designing any ML model.

`GridSearchCV()` is a function in `sklearn` which determines the best combination of parameters for a model, and should be looked into.

## 6. XGBoost

Previous examples have used a RandomForest model, which averages the predictions of many decision trees.

RandomForest is an ensemble method, of which another is **gradient boosting**.

### Gradient Boosting

Gradient boosting iterates through cycles to add models to an ensemble.

An ensemble is initialised with a single model, whose predictons are relatively naive.

- The current ensemble generates predictions for each observation in the dataset. Predictions from all models in the ensemble are added together.
- These predictions are used to calculate a loss function e.g. MSE
- The loss function is used to fit a new model that is added to the ensemble. Specifically, model parameters are determined such that adding the model will reduce loss.
- The new model is added to the ensemble and the cycle repeats.

Gradient descent is used on the loss function to determine the parameters in the new model.

### Example

In [9]:
# Load data
df = pd.read_csv(MELB)

# Define predictors
list_columns = ['Rooms', 'Distance', 'Landsize', 'BuildingArea', 'YearBuilt']
# Extract data
X = df[list_columns]

# Select target
y = df['Price']

# Train/test split
X_train, X_valid, y_train, y_valid = train_test_split(X, y)

# Show data
X_train.head()

,Rooms,Distance,Landsize,BuildingArea,YearBuilt
9967,3,31.6,2385.0,127.0,1930.0
8513,4,8.0,551.0,NaN,NaN
1614,3,7.8,692.0,164.0,1940.0
115,2,3.3,79.0,68.0,1890.0
11793,3,5.1,0.0,NaN,2005.0


We will work with the `XGBoost` (extreme gradient boosting) library, which implements gradient boosting with additional features based on performance and speed.

`sklearn` has another version of gradient boosting, but XGBoost has technical advantages.

In [10]:
# Initialise model
my_model = XGBRegressor()
# Fit model
my_model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [11]:
# Make predictions with model
y_pred = my_model.predict(X_valid)

# Calculate MAE
mae = mean_absolute_error(y_valid, y_pred)

# Report
print(f'Mean Absolute Error: {mae:,.2f}')

Mean Absolute Error: 244,737.49


### Parameter Tuning

XGBoost has parameters that can dramatically affect accuracy and training speed.

#### `n_estimators`

This specifies how many times to go through the modeling cycle described above. It is equal to the number of models that will be included in the ensemble.
- Too low a value causes underfitting, leading to inaccurate predictions on training and test data
- Too high a value causes overfitting, causing accurate predictions on training data, but inaccurate predictions on test data

Typical values range from 100-1000; however, this depends on the `learning_rate` parameter.

In [15]:
# Initialise model
my_model = XGBRegressor(n_estimators = 250)
# Fit model 
my_model.fit(X_train, y_train)
# Make predictions with model
y_pred = my_model.predict(X_valid)
# Calculate MAE
mae = mean_absolute_error(y_valid, y_pred)

# Report
print(f'Mean Absolute Error: {mae:,.2f}')

Mean Absolute Error: 248,496.46


#### `early_stopping_rounds`

This offers a way to automatically find the ideal value for n_estimators. The model stops iterating when the validation score stops improving, even when the max `n_estimators` has not yet been reached.

It is prudent to set a high `n_estimators` value and then use `early_stopping_rounds` to find the optimal time to stop iterating.

Random chance can cause a single round where validation scores don't improve, so a number is specified for rounds of no deterioration to allow before stopping. Setting `early_stopping_rounds = 5` is a reasonable choice.

When using `early_stopping_rounds`, you must also set aside data for validation scores, done using the `eval_set` parameter.

In [20]:
# Initialise model
my_model = XGBRegressor(n_estimators = 500, early_stopping_rounds = 5)
# Fit model
my_model.fit(
    X_train, y_train,
    eval_set = [(X_valid, y_valid)],
    verbose = False
)
# Make predictions with model
y_pred = my_model.predict(X_valid)
# Calculate MAE
mae = mean_absolute_error(y_valid, y_pred)

# Report
print(f'Mean Absolute Error: {mae:,.2f}')

Mean Absolute Error: 251,713.90


#### `learning_rate`

Instead of getting predictions by sum'ing predictions from each component model, we can multiply predictions from each model by a small number (the learning rate) before adding them in.

This means each tree we add to the ensemble helps us less, allowing a higher `n_estimators` value before overfitting.

In general, a small `learning_rate` with a large `n_estimators` will yield more accurate models. However, it will take the model longer to train since it does more iterations through the cycle. XGBoost defaults to `learning_rate = 0.1`.

In [23]:
# Initialise model
my_model = XGBRegressor(n_estimators = 1000, early_stopping_rounds  = 5, learning_rate = 0.05)
# Fit model to data
my_model.fit(
    X_train, y_train,
    eval_set = [(X_valid, y_valid)],
    verbose = False
)

# Make predictions with model
y_pred = my_model.predict(X_valid)
# Calculate MAE
mae = mean_absolute_error(y_valid, y_pred)

# Report
print(f'Mean Absolute Error: {mae:,.2f}')

Mean Absolute Error: 242,697.75


#### `n_jobs`

On larger datasets where runtime is a consideration, parallelism can be used to build models faster. It is common to set the parameter `n_jobs` equal to the number of cores on your machine.

The resulting model won't be any better, but its useful for large datasets where you would wait for the `.fit()` command.

In [24]:
# Initialise model
my_model = XGBRegressor(
    n_estimators = 1000, 
    early_stopping_rounds = 5, 
    learning_rate = 0.05,
    n_jobs = 4
)

# Fit model
my_model.fit(
    X_train, y_train,
    eval_set = [(X_valid, y_valid)],
    verbose = False
)

# Make predictions with model
y_pred = my_model.predict(X_valid)
# Calculate MAE
mae = mean_absolute_error(y_valid, y_pred)

# Report
print(f'Mean Absolute Error: {mae:,.2f}')

Mean Absolute Error: 242,697.75


XGBoost is the leading software library for working with standard tabular data as opposed to images/video.

## 7. Data Leakage

Data leakage happens when the training data contains information about the target, but similar data is not available when the model is used for prediction.

This leads to high performance on the training set (and possibly validation data), but the model performs poorly in production.

Leakage causes models to look accurate until decisions are made with the model.

There are two main types of leakage; **target leakage**, and **train-test contamination**.

### Target Leakage

Target leakage occurs when the predictors include data not available at the prediction stage. This is important in terms of the chronological order that data becomes available, not only if the feature helps make good predictions.

Consider the following example:

| got_pneumonia | age | weight | male  | took_antibiotic_medicine | ... |
|---------------|-----|--------|-------|---------------------------|-----|
| False         | 65  | 100    | False | False                     | ... |
| False         | 72  | 130    | True  | False                     | ... |
| True          | 58  | 100    | False | True                      | ... |

Patients take antibiotics _after_ getting pneumonia in order to recover. Raw data shows a strong relationship between these columns; however `took_antibiotic_medicine` is frequently changed _after_ the value for `got_pneumonia` is determined. This is target leakage.

The model would see that anyone with a value of `False` for `took_antibiotic_medicine` didn't have pneumonia. Since validation data comes from the same source as training data, the pattern will repeat itself in validation, leading to excellent cross/validation scores.

However, the model would be inaccurate when deployed in the real world. Patients who _will_ get pneumonia won't have received antibiotics yet.

To prevent this kind of leakage, any variable updated (or created) **after target value is realised** should be excluded.

### Train-Test Contamination

This occurs when there is insufficient distinguishing between training data and validation data.

Recall that validation is meant to be a measure of how the model does on data not before considered. You can corrupt this process in subtle ways if teh validation data affects the preprocessing behaviour.

For example, you run an Imputer before calling `train_test_split()`. The model will get good validation scores, but perform poorly when deployed.

Data was incorporated from the validation or test data when considering predictions, as imputation took into account all data, before training data was generated. This problem becomes more subtle when undertaking more complex feature engineering.

If validation is based on a train/test split, exclude the validation data from any kind of fitting, including the preprocessing steps. This is easy when using `sklearn` pipelines. When using cross-validation it's more critical to do preprocessing inside the pipeline.

### Example

We will learn to detect and remove target leakage using the `credit_card_data.csv` dataset.

In [29]:
# Load data
df = pd.read_csv(AER, true_values = ['yes'], false_values = ['no'])

# Select target
y = df['card']

# Select predictors
X = df.drop(columns = 'card')

# Report
print(f'Number of rows in dataset: {X.shape[0]}')
# Show data
X.head()

Number of rows in dataset: 1319


,reports,age,income,share,expenditure,owner,selfemp,dependents,months,majorcards,active
0,0,37.66667,4.5200,0.033270,124.983300,True,False,3,54,1,12
1,0,33.25000,2.4200,0.005217,9.854167,False,False,3,34,1,13
2,0,33.66667,4.5000,0.004156,15.000000,True,False,4,58,1,5
3,0,30.50000,2.5400,0.065214,137.869200,False,False,0,25,1,7
4,0,32.16667,9.7867,0.067051,546.503300,True,False,2,64,1,5


In [35]:
# Generate pipeline
my_pipeline = make_pipeline(
    RandomForestClassifier(n_estimators = 100)
)

# Run cross validation
cv_scores = cross_val_score(
    my_pipeline, X, y,
    cv = 5,
    scoring = 'accuracy'
)

# Report
print(f'Cross-validation accuracy: {cv_scores.mean():.2f}')

Cross-validation accuracy: 0.98


Generally speaking, it's rare to find models that are 98% accurate, and so any seemingly high accuracy should be inspected for target leakage.

The `expenditure` variable is poorly explained. It is unclear if it is expenditure on the relevant card or cards used before application.

In [41]:
# Get expenditure values
expenditures_card = X['expenditure'][y]
expenditures_nocard = X['expenditure'][~y]

# No card and no expenditures
nocard_noexp = (expenditures_nocard == 0).mean()
# Card and no expenditures
card_noexp = (expenditures_card == 0).mean()
# Calculate percentages
percent_ncne = nocard_noexp * 100
percent_ycne = card_noexp * 100

# Report
print(f'Fraction of those who did not receive a card and had no expenditures: {percent_ncne:.2f}%')
print(f'Fraction of those who received a card and had no expenditures: {percent_ycne:.2f}%')

Fraction of those who did not receive a card and had no expenditures: 100.00%
Fraction of those who received a card and had no expenditures: 2.05%


100% of those **not** receiving a card had no expenditures, while only 2% of those who **did receive** a card had no expenditures. This is a case of target leakage as expenditures probably occurred _after_ the card was applied for.

The `share` variable is partially determined by `expenditure` and should also be excluded. The `active` and `majorcards` variables are less clear, but may be an issue. If it is not possible to track down the originators of the data to confirm the meaning of variables, it is better to be safe than sorry.